In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
import numpy as np
from datasets import load_dataset, concatenate_datasets

# Exercise 7: Multi-head Attention

# Loading data

In [26]:
from gensim.downloader import load as gensim_load

glove = gensim_load("glove-wiki-gigaword-100")

In [27]:
full_ds_dict = load_dataset("imdb")
ds_train_full = full_ds_dict["train"]
ds_test_full = full_ds_dict["test"]

In [28]:
from datasets import Dataset as HFDataset


def sample_imdb(ds: HFDataset, n_samples: int) -> HFDataset:
    """
    Sample n_samples from the imdb dataset, ensuring that the dataset is balanced.
    """
    # How many samples per class
    n_per_class = n_samples // 2

    # Filter each class
    pos = (
        ds.filter(lambda x: x["label"] == 1).shuffle(seed=42).select(range(n_per_class))
    )
    neg = (
        ds.filter(lambda x: x["label"] == 0).shuffle(seed=42).select(range(n_per_class))
    )

    # Combine and shuffle
    balanced = concatenate_datasets([pos, neg]).shuffle(seed=42)

    return balanced


In [29]:
TRAIN_SAMPLES = 1000

assert isinstance(ds_train_full, HFDataset)
ds_train = sample_imdb(ds_train_full, TRAIN_SAMPLES)

assert (
    ds_train.filter(lambda x: x["label"] == 1).num_rows
    == ds_train.filter(lambda x: x["label"] == 0).num_rows
)

In [30]:
ds_train

Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})

# Tokenization

In [31]:
import re


def tokenize(review: str) -> list[str]:
    # Convert to lowercase
    text = review.lower()

    # Remove all numbers entirely from the text
    text = re.sub(r"\d+", "", text)

    # on word boundaries and ignores punctuation
    tokens: list[str] = re.findall(r"\w+", text)

    return tokens


sample_sentence = "This is a sentence with an unknown word: supermuel"

tokenized = tokenize(sample_sentence)
tokenized[:15]

['this', 'is', 'a', 'sentence', 'with', 'an', 'unknown', 'word', 'supermuel']

In [32]:
from typing import Any

UNK_TOKEN = "<unk>"


def remove_tokens_not_in_glove(
    tokens: list[str],
    glove: Any,
    unk_token: str = UNK_TOKEN,
) -> list[str]:
    return [token if token in glove else unk_token for token in tokens]


PAD_TOKEN = "<pad>"


def pad_or_truncate(
    tokens: list[str],
    sequence_length: int,
    pad_token: str = PAD_TOKEN,
) -> list[str]:
    truncated = tokens[:sequence_length]
    return truncated + [pad_token] * (sequence_length - len(truncated))


with_tokens_removed = remove_tokens_not_in_glove(tokenized, glove=glove)
print(with_tokens_removed)

padded = pad_or_truncate(with_tokens_removed, sequence_length=15)
print(padded)

['this', 'is', 'a', 'sentence', 'with', 'an', 'unknown', 'word', '<unk>']
['this', 'is', 'a', 'sentence', 'with', 'an', 'unknown', 'word', '<unk>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']


In [33]:
def preprocess_review(
    review: str,
    sequence_length: int,
    glove: Any,
    unk_token: str = UNK_TOKEN,
    pad_token: str = PAD_TOKEN,
) -> list[str]:
    tokens = tokenize(review)
    tokens = remove_tokens_not_in_glove(tokens, glove, unk_token)
    tokens = pad_or_truncate(tokens, sequence_length, pad_token)
    return tokens


preprocess_review(sample_sentence, sequence_length=15, glove=glove)

['this',
 'is',
 'a',
 'sentence',
 'with',
 'an',
 'unknown',
 'word',
 '<unk>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>']

## Tokenize IMDB

In [34]:
TRAIN_SAMPLES = 1000
ds_train = sample_imdb(ds_train, TRAIN_SAMPLES)


def pre_process_dataset(
    ds: HFDataset,
    sequence_length: int,
    glove: Any,
    unk_token: str = UNK_TOKEN,
    pad_token: str = PAD_TOKEN,
) -> tuple[HFDataset, list[str], dict[str, int], dict[int, str]]:
    ds = ds.map(
        lambda x: {
            "tokens": preprocess_review(
                x["text"], sequence_length, glove, unk_token, pad_token
            )
        }
    )

    vocab: list[str] = sorted(
        list(set(token for review in ds["tokens"] for token in review))
    )

    if PAD_TOKEN not in vocab:
        vocab.insert(0, PAD_TOKEN)
    if UNK_TOKEN not in vocab:
        vocab.insert(1, UNK_TOKEN)

    assert vocab[0] == PAD_TOKEN
    assert vocab[1] == UNK_TOKEN

    word_to_idx = {word: i for i, word in enumerate(vocab)}
    idx_to_word = {i: word for word, i in word_to_idx.items()}

    ds = ds.map(lambda x: {"input_ids": [word_to_idx[token] for token in x["tokens"]]})

    return ds, vocab, word_to_idx, idx_to_word


ds_train, vocab, word_to_idx, idx_to_word = pre_process_dataset(
    ds_train, sequence_length=15, glove=glove
)
ds_train

Map: 100%|██████████| 1000/1000 [00:00<00:00, 36024.25 examples/s]


Dataset({
    features: ['text', 'label', 'tokens', 'input_ids'],
    num_rows: 1000
})

In [35]:
ds_train[0]["tokens"][:15], ds_train[0]["input_ids"][:15]

(['the',
  'cast',
  'of',
  'this',
  'film',
  'contain',
  'some',
  'of',
  'new',
  'zealander',
  's',
  'better',
  'actors',
  'many',
  'of'],
 [2810,
  431,
  1969,
  2831,
  1057,
  589,
  2629,
  1969,
  1910,
  3174,
  2421,
  291,
  30,
  1726,
  1969])

In [36]:
PAD_TOKEN_ID = word_to_idx[PAD_TOKEN]
PAD_TOKEN_ID

0

In [37]:
print(f"Vocabulary size: {len(word_to_idx)}")

Vocabulary size: 3181


# Vectorize

In [38]:
def create_embedding_matrix(glove: Any, word_to_idx: dict[str, int]) -> torch.Tensor:
    """
    Create an embedding matrix from a GloVe model and a word-to-index mapping.

    Args:
        glove: The GloVe model instance.
        word_to_idx: Mapping from word to index.

    Returns:
        A torch.Tensor of shape (vocab_size, embedding_dim).
    """
    embedding_dim: int = glove.vector_size  # type: ignore
    embedding_matrix = np.zeros((len(word_to_idx), embedding_dim))
    for word, i in word_to_idx.items():
        if word in glove:
            embedding_matrix[i] = glove[word]  # type: ignore
        # else, it remains a zero vector (for <unk>, <pad>, etc.)
    embedding_matrix_tensor = torch.tensor(embedding_matrix, dtype=torch.float32)
    return embedding_matrix_tensor


embedding_matrix = create_embedding_matrix(glove, word_to_idx)
print(f"{embedding_matrix.shape=}")

embedding_matrix.shape=torch.Size([3181, 100])


In [39]:
class IMDBReviewDataset(torch.utils.data.Dataset):
    """
    Custom PyTorch Dataset for the IMDB reviews.
    It takes a Hugging Face Dataset object and prepares items for PyTorch.
    """

    def __init__(self, hf_dataset: HFDataset):
        self.hf_dataset = hf_dataset
        if "input_ids" not in self.hf_dataset.column_names:
            raise ValueError("input_ids column not found in dataset")
        if "label" not in self.hf_dataset.column_names:
            raise ValueError("label column not found in dataset")

    def __len__(self):
        """Returns the total number of samples in the dataset."""
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        """
        Fetches the sample at the given index and converts it to PyTorch tensors.
        """
        # Get the sample from the Hugging Face dataset
        sample = self.hf_dataset[idx]

        # Extract input_ids and label
        input_ids = sample["input_ids"]
        label = sample["label"]

        # Convert to PyTorch Tensors
        # input_ids should be LongTensor for the embedding layer
        input_tensor = torch.tensor(input_ids, dtype=torch.long)

        # label should be a FloatTensor for the loss function (e.g., BCEWithLogitsLoss)
        # We also add a dimension to make its shape [1] instead of a scalar
        label_tensor = torch.tensor(label, dtype=torch.float32).unsqueeze(0)

        return input_tensor, label_tensor


# Create an instance of our custom dataset
pytorch_train_dataset = IMDBReviewDataset(ds_train)

# Let's check one sample
sample_input, sample_label = pytorch_train_dataset[0]
print(f"Sample input tensor shape: {sample_input.shape}")
print(f"Sample label tensor shape: {sample_label.shape}")
print("-" * 20)

Sample input tensor shape: torch.Size([15])
Sample label tensor shape: torch.Size([1])
--------------------


# Multi-Head Self Attention

In [ ]:
from math import sqrt


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, n_heads: int, d_model: int):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model = d_model
        self.n_heads = n_heads

        self.d_k = d_model // n_heads
        assert self.d_k > 1

        # In normal self-attention, W_Q is of shape (d_model, d_k)
        # But here, we want n W_Q matrixes. Since we also want to
        # keep the computational needs similar to the original self-attention,
        # we'll use n **smaller** W_Q matrixes. A simple rule is to use
        # d_k = d_model // n_heads

        # But how to define n_heads matrixes ? Well, can we cheat and instead
        # create a bigger one that act as multiple smaller ones ?
        # We need n_heads matrixes of size (d_model, d_model // n_heads)
        # So this is exactly the same as one matrix of size (d_model, d_model)

        self.W_Q = nn.Linear(d_model, d_model)

        # Exactly the same for W_K
        self.W_K = nn.Linear(d_model, d_model)

        # In the normal self-attention, W_V is of size (d_model, d_v)
        # For simplicity, d_v is often set to d_k. So we can define
        # W_V of size (d_model, d_model) like the others
        self.W_V = nn.Linear(d_model, d_model)

        # final projection matrix
        self.W_O = nn.Linear(in_features=d_model, out_features=self.d_model)

    def forward(self, sentences: torch.Tensor, mask:torch.Tensor | None = None):
        # sentences is of shape (batch_size, sentence_length, d_model)
        batch_size, sentence_length, d_model = sentences.shape
        n_heads, d_k = self.n_heads, self.d_k

        assert d_model == self.d_model

        Q: torch.Tensor = self.W_Q(sentences)  # (batch_size, sentence_length, d_model)
        K: torch.Tensor = self.W_K(sentences)  # (batch_size, sentence_length, d_model)
        V: torch.Tensor = self.W_V(sentences)  # (batch_size, sentence_length, d_model)

        assert Q.shape == (batch_size, sentence_length, d_model)
        assert K.shape == (batch_size, sentence_length, d_model)
        assert V.shape == (batch_size, sentence_length, d_model)

        Q = Q.view(batch_size, sentence_length, n_heads, d_k)
        K = K.view(batch_size, sentence_length, n_heads, d_k)
        V = V.view(batch_size, sentence_length, n_heads, d_k)

        # Bring n_heads dimension up
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        assert Q.shape == (batch_size, n_heads, sentence_length, d_k)
        assert K.shape == (batch_size, n_heads, sentence_length, d_k)
        assert V.shape == (batch_size, n_heads, sentence_length, d_k)

        # Now we need to compute the raw scores by multiplying Q and K
        # The result should be of shape (batch_size, n_heads, sentence_length, sentence_length)
        # For that, we need to multiply something of shapes:
        # (batch_size, n_heads, sentence_length, d_k) (batch_size, n_heads, d_k, sentence_length)

        raw_scores = Q @ K.transpose(-1, -2)
        assert raw_scores.shape == (
            batch_size,
            n_heads,
            sentence_length,
            sentence_length,
        )

        # Now we scale
        scaled_scores = raw_scores / sqrt(d_k)
               

        # Masking
        if mask is not None:
            # The mask should already be broadcastable, e.g., (batch, 1, 1, seq_len)
            masked_scores = scaled_scores.masked_fill(mask, float("-inf"))
        else:
            masked_scores = scaled_scores


        # Compute the attention scores
        attention_scores = F.softmax(masked_scores, dim=-1)
        assert attention_scores.shape == (
            batch_size,
            n_heads,
            sentence_length,
            sentence_length,
        )  # Shape should not change

        # Get the values
        # Outputs should be of shape (batch_size, n_heads, sentence_length, d_k)
        outputs = attention_scores @ V
        # (batch_size, n_heads, sentence_length,sentence_length) @ (batch_size, n_heads, sentence_length, d_k)
        # = (batch_size, n_heads, sentence_length, d_k)
        assert outputs.shape == (batch_size, n_heads, sentence_length, d_k)

        # concatenate outputs into (batch_size, sentence_length, d_model)
        outputs = (
            outputs.transpose(1, 2)
            .contiguous()
            .view(batch_size, sentence_length, d_model)
        )

        # Apply last linear transformation
        return self.W_O(outputs)


In [42]:
from torch.nn.functional import relu


class PointWiseFeedForward(nn.Module):
    def __init__(self, d_model: int, inner_dim: int | None = None):
        super().__init__()
        if not inner_dim:
            # It's common to have a larger inner dimension
            inner_dim = d_model * 4
            self.inner_dim = inner_dim

        self.linear_1 = nn.Linear(d_model, self.inner_dim)
        self.linear_2 = nn.Linear(self.inner_dim, d_model)

    def forward(self, x):
        x = self.linear_1(x)
        x = relu(x)
        x = self.linear_2(x)
        return x


class Encoder(nn.Module):
    def __init__(self, n_heads: int, d_model: int, dropout_p: float = 0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model

        self.multi_head_self_attention = MultiHeadSelfAttention(
            n_heads=n_heads, d_model=d_model
        )
        self.layer_norm1 = nn.LayerNorm(normalized_shape=self.d_model)
        self.ff = PointWiseFeedForward(d_model=d_model)
        self.layer_norm2 = nn.LayerNorm(normalized_shape=self.d_model)

        # Define dropout layers
        self.dropout1 = nn.Dropout(dropout_p)
        self.dropout2 = nn.Dropout(dropout_p)

    def forward(self, inputs: torch.Tensor,  mask: torch.Tensor | None = None):
        batch_size, sentence_length, d_model = inputs.shape
        assert d_model == self.d_model

        attn_output = self.multi_head_self_attention(inputs, mask=mask)
        before_ffn = self.layer_norm1(inputs + self.dropout1(attn_output))
        ffn_output = self.ff(before_ffn)
        outputs = self.layer_norm2(before_ffn + self.dropout2(ffn_output))

        return outputs

## Classification Layer

In [43]:
class BinaryClassificationModel(nn.Module):
    def __init__(
        self,
        embedding_matrix: torch.Tensor,
        d_model: int,
        n_heads: int,
        freeze_embedding: bool = False,
    ) -> None:
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix, freeze=freeze_embedding
        )

        self.d_model = d_model
        self.n_heads = n_heads

        self.encoder = Encoder(n_heads=n_heads, d_model=d_model)

        self.linear = nn.Linear(in_features=d_model, out_features=1)

    def forward(self, inputs):
        # inputs is of shape (batch_size, sequence_length)

        # Step 1: get the embeddings

        embeddings = self.embedding(inputs)
        # shape of embeddings (batch_size, sequence_length, d_model)

        pad_mask = (inputs == PAD_TOKEN_ID) # Shape: (batch, seq_len)
    
        # Reshape for broadcasting in the attention module
        # Shape: (batch, 1, 1, seq_len)
        attention_mask = pad_mask.unsqueeze(1).unsqueeze(2)

        # Step 2: call encoder
        attention_output = self.encoder(embeddings, mask=attention_mask)
        # shape of attention_output is of [batch_size, sequence_length, d_k]

        # Step 3: apply linear
        pooled_output = torch.mean(attention_output, dim=1)
        # (batch_size, d_k)

        logits = self.linear(pooled_output)

        return logits

# Training

### Training dataset

In [44]:
TRAIN_SAMPLES = 25000
SEQUENCE_LENGTH = 512

assert isinstance(ds_train_full, HFDataset)
ds_train_unpreprocessed = sample_imdb(ds_train_full, TRAIN_SAMPLES)

ds_train, vocab, word_to_idx, idx_to_word = pre_process_dataset(
    ds_train_unpreprocessed, sequence_length=SEQUENCE_LENGTH, glove=glove
)

print(f"Samples: {len(ds_train)}")
print(f"Vocab size: {len(vocab)}")

embedding_matrix = create_embedding_matrix(glove, word_to_idx)
print(f"{embedding_matrix.shape=}")

pytorch_train_dataset = IMDBReviewDataset(ds_train)

# Check one sample
sample_input, sample_label = pytorch_train_dataset[0]
print(f"Sample input tensor shape: {sample_input.shape}")
print(f"Sample label tensor shape: {sample_label.shape}")

Map: 100%|██████████| 25000/25000 [00:06<00:00, 3719.72 examples/s]


Samples: 25000
Vocab size: 58549
embedding_matrix.shape=torch.Size([58549, 100])
Sample input tensor shape: torch.Size([512])
Sample label tensor shape: torch.Size([1])


### Testing dataset

In [45]:
# 1. Create a balanced test sample
TEST_SAMPLES = 1000
assert isinstance(ds_test_full, HFDataset)
ds_test_unpreprocessed = sample_imdb(ds_test_full, TEST_SAMPLES)


# 2. Pre-process the test dataset
def process_test_data(
    ds: HFDataset, word_to_idx: dict[str, int], sequence_length: int, glove: Any
) -> HFDataset:
    # This simplified function processes text using the *existing* vocabulary
    def text_to_ids(example):
        tokens = preprocess_review(example["text"], sequence_length, glove)
        # Map tokens to indices, using the <unk> index for words not in the training vocab
        unk_idx = word_to_idx[UNK_TOKEN]
        input_ids = [word_to_idx.get(token, unk_idx) for token in tokens]
        return {"input_ids": input_ids}

    ds = ds.map(text_to_ids)
    return ds


ds_test = process_test_data(ds_test_unpreprocessed, word_to_idx, SEQUENCE_LENGTH, glove)

# 3. Create the PyTorch Dataset and DataLoader for testing
pytorch_test_dataset = IMDBReviewDataset(ds_test)


def _manual_f1_score(y_true: list[float], y_pred: list[float]) -> float:
    """Compute F1 score manually for binary classification."""
    # Convert to int for safety
    y_true_int = [int(round(x)) for x in y_true]
    y_pred_int = [int(round(x)) for x in y_pred]
    tp = sum(1 for yt, yp in zip(y_true_int, y_pred_int) if yt == 1 and yp == 1)
    fp = sum(1 for yt, yp in zip(y_true_int, y_pred_int) if yt == 0 and yp == 1)
    fn = sum(1 for yt, yp in zip(y_true_int, y_pred_int) if yt == 1 and yp == 0)
    # Precision and recall
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    # F1
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def evaluate_model(
    model: nn.Module,
    dataloader: torch.utils.data.DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float, float]:
    model.eval()  # Set model to evaluation mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    all_labels: list[float] = []
    all_preds: list[float] = []

    with torch.no_grad():  # Disable gradient calculation
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.float().to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # --- Calculate Accuracy ---
            # Get probabilities from logits
            probs = torch.sigmoid(outputs)
            # Get predictions (0 or 1) by rounding
            predicted = torch.round(probs)

            # Update counters
            running_loss += loss.item() * inputs.size(0)
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            # Store for F1 calculation
            all_labels.extend(labels.detach().cpu().numpy().flatten().tolist())
            all_preds.extend(predicted.detach().cpu().numpy().flatten().tolist())

    avg_loss = float(running_loss / total_samples)
    accuracy = float(correct_predictions / total_samples)
    f1 = float(_manual_f1_score(all_labels, all_preds))
    return avg_loss, accuracy, f1

Map: 100%|██████████| 1000/1000 [00:00<00:00, 7291.44 examples/s]


### Actual training

In [46]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score
import copy


def train_and_evaluate_model(
    model: nn.Module,
    train_dataset: torch.utils.data.Dataset,
    test_dataset: torch.utils.data.Dataset,
    epochs: int,
    batch_size: int,
    lr: float,
    device: torch.device,
    patience: int = 3,
) -> nn.Module:
    model.to(device)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_f1 = float("-inf")
    epochs_no_improve = 0

    try:
        for epoch in range(epochs):
            # --- Training Phase ---
            model.train()
            train_running_loss = 0.0
            correct_train_predictions = 0
            total_train_samples = 0
            all_train_labels: list[float] = []
            all_train_preds: list[float] = []
            for inputs, labels in train_dataloader:
                inputs = inputs.to(device)
                labels = labels.float().to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                train_running_loss += loss.item() * inputs.size(0)

                # Calculate training accuracy and F1
                probs = torch.sigmoid(outputs)
                predicted = torch.round(probs)
                correct_train_predictions += (predicted == labels).sum().item()
                total_train_samples += labels.size(0)

                all_train_labels.extend(
                    labels.detach().cpu().numpy().flatten().tolist()
                )
                all_train_preds.extend(
                    predicted.detach().cpu().numpy().flatten().tolist()
                )

            epoch_train_loss = train_running_loss / len(train_dataset)  # type: ignore
            epoch_train_acc = correct_train_predictions / total_train_samples
            epoch_train_f1 = f1_score(all_train_labels, all_train_preds)

            # --- Evaluation Phase ---
            epoch_test_loss, epoch_test_acc, epoch_test_f1 = evaluate_model(
                model, test_dataloader, criterion, device
            )

            print(
                f"Epoch {epoch + 1}/{epochs} | "
                f"Train Loss: {epoch_train_loss:.4f} | "
                f"Train Acc: {epoch_train_acc:.4f} | "
                f"Train F1: {epoch_train_f1:.4f} | "
                f"Test Loss: {epoch_test_loss:.4f} | "
                f"Test Acc: {epoch_test_acc:.4f} | "
                f"Test F1: {epoch_test_f1:.4f}"
            )

            # --- Early Stopping Check ---
            if epoch_test_f1 > best_f1:
                best_f1 = epoch_test_f1
                best_model_wts = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(
                        f"Early stopping at epoch {epoch + 1}. Best Test F1: {best_f1:.4f}"
                    )
                    break
    except KeyboardInterrupt:
        print("Training interrupted by user. Returning best model so far.")

    # Load best model weights before returning
    model.load_state_dict(best_model_wts)
    return model


In [47]:
D_MODEL = embedding_matrix.shape[1]
N_HEADS = 5
EPOCHS = 20
BATCH_SIZE = 64
LR = 1e-3

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device {DEVICE}")

model = BinaryClassificationModel(
    embedding_matrix=embedding_matrix,
    d_model=D_MODEL,
    n_heads=N_HEADS,
)
print(model)

# --- Finally, update your function call ---
model = train_and_evaluate_model(
    model=model,
    train_dataset=pytorch_train_dataset,
    test_dataset=pytorch_test_dataset,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    device=DEVICE,
)

Using device mps
BinaryClassificationModel(
  (embedding): Embedding(58549, 100)
  (encoder): Encoder(
    (multi_head_self_attention): MultiHeadSelfAttention(
      (W_Q): Linear(in_features=100, out_features=100, bias=True)
      (W_K): Linear(in_features=100, out_features=100, bias=True)
      (W_V): Linear(in_features=100, out_features=100, bias=True)
      (W_O): Linear(in_features=100, out_features=100, bias=True)
    )
    (layer_norm1): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
    (ff): PointWiseFeedForward(
      (linear_1): Linear(in_features=100, out_features=400, bias=True)
      (linear_2): Linear(in_features=400, out_features=100, bias=True)
    )
    (layer_norm2): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (linear): Linear(in_features=100, out_features=1, bias=True)
)
scaled_scores.shape=torch.Size([64, 5, 512, 512])
scaled_scores.shape=torch.Size([6

# Inference

In [ ]:
def predict_sentiment(
    text: str,
    model: nn.Module,
    word_to_idx: dict[str, int],
    sequence_length: int,
    glove: Any,
    device: torch.device,
) -> str:
    """
    Predicts the sentiment of a single text string using the trained model.
    """
    # 1. Set model to evaluation mode
    model.eval()

    # 2. Preprocess the text
    # This pipeline should be identical to the one used for training/testing
    tokens = preprocess_review(text, sequence_length, glove)

    # Map tokens to indices, using the <unk> index for words not in the training vocab
    unk_idx = word_to_idx[UNK_TOKEN]
    input_ids = [word_to_idx.get(token, unk_idx) for token in tokens]

    # 3. Convert to tensor and add batch dimension
    input_tensor = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)
    # Shape is now [1, sequence_length]

    # 4. Run inference
    with torch.no_grad():
        output = model(input_tensor)

    # 5. Interpret the output
    prob = torch.sigmoid(output).item()  # Get probability between 0 and 1
    prediction = round(prob)  # Round to get 0 or 1

    # 6. Return human-readable result
    print(f"Input Text: '{text}'")
    print(f"Model Logit: {output.item():.4f}, Probability: {prob:.4f}")

    if prediction == 1:
        return "Positive"
    else:
        return "Negative"


In [ ]:
# Example 1
sentiment1 = predict_sentiment(
    "This movie was fantastic and full of suspense!",
    model,
    word_to_idx,
    SEQUENCE_LENGTH,
    glove,
    DEVICE,
)
print(f"Predicted Sentiment: {sentiment1}\n")


# Example 2
sentiment2 = predict_sentiment(
    "This movie was a complete waste of my time, truly bad.",
    model,
    word_to_idx,
    SEQUENCE_LENGTH,
    glove,
    DEVICE,
)
print(f"Predicted Sentiment: {sentiment2}\n")


# Example 3 (A more neutral/tricky one)
sentiment3 = predict_sentiment(
    "The acting was okay but the plot was a little predictable.",
    model,
    word_to_idx,
    SEQUENCE_LENGTH,
    glove,
    DEVICE,
)
print(f"Predicted Sentiment: {sentiment3}\n")

Input Text: 'This movie was fantastic and full of suspense!'
Model Logit: 0.5176, Probability: 0.6266
Predicted Sentiment: Positive

Input Text: 'This movie was a complete waste of my time, truly bad.'
Model Logit: -0.2961, Probability: 0.4265
Predicted Sentiment: Negative

Input Text: 'The acting was okay but the plot was a little predictable.'
Model Logit: -0.3990, Probability: 0.4016
Predicted Sentiment: Negative



# Finding Wrong Predictions

Let's analyze the test dataset to find examples where our model made incorrect predictions.


In [ ]:
# FIX: Find ALL wrong predictions, not just the first 15
print("FINDING ALL WRONG PREDICTIONS:")
print("=" * 40)


def find_all_wrong_predictions(
    test_dataset: torch.utils.data.Dataset,
    device: torch.device,
) -> list[dict[str, str | int | float]]:
    """
    Find ALL examples where the model made incorrect predictions.
    """
    model.eval()
    wrong_predictions = []

    # Create a dataloader for the test dataset
    test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    with torch.no_grad():
        for idx, (inputs, labels) in enumerate(test_dataloader):
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Get model prediction
            outputs = model(inputs)
            prob = torch.sigmoid(outputs).item()
            predicted = int(round(prob))
            actual = int(labels.item())

            # Check if prediction is wrong
            if predicted != actual:
                # Get the original text from the test dataset
                original_text = ds_test[idx]["text"]

                wrong_predictions.append(
                    {
                        "index": idx,
                        "text": original_text,
                        "actual_label": actual,
                        "predicted_label": predicted,
                        "confidence": prob,
                        "actual_sentiment": "Positive" if actual == 1 else "Negative",
                        "predicted_sentiment": "Positive"
                        if predicted == 1
                        else "Negative",
                    }
                )

    return wrong_predictions


# Find ALL wrong predictions
all_wrong_preds = find_all_wrong_predictions(
    model, pytorch_test_dataset, torch.device(DEVICE)
)

print(
    f"Found {len(all_wrong_preds)} wrong predictions out of {len(pytorch_test_dataset)} test samples"
)
correct_accuracy = (
    (len(pytorch_test_dataset) - len(all_wrong_preds)) / len(pytorch_test_dataset) * 100
)
print(f"Correct model accuracy: {correct_accuracy:.2f}%")

print(f"\nPrevious analysis error:")
print(f"  I only found the first 15 wrong predictions")
print(f"  But calculated accuracy as if I found ALL wrong predictions")
print(
    f"  This gave me: {(1000 - 15) / 1000 * 100:.1f}% instead of {correct_accuracy:.1f}%"
)

print(f"\nVerification:")
print(f"  Manual calculation: 87.80%")
print(f"  Training evaluation: 87.80%")
print(f"  Corrected analysis: {correct_accuracy:.2f}%")
print("✓ All calculations now match!")


FINDING ALL WRONG PREDICTIONS:
Found 122 wrong predictions out of 1000 test samples
Correct model accuracy: 87.80%

Previous analysis error:
  I only found the first 15 wrong predictions
  But calculated accuracy as if I found ALL wrong predictions
  This gave me: 98.5% instead of 87.8%

Verification:
  Manual calculation: 87.80%
  Training evaluation: 87.80%
  Corrected analysis: 87.80%
✓ All calculations now match!


In [ ]:
# CORRECTED ANALYSIS: Length analysis with ALL wrong predictions
print("CORRECTED LENGTH ANALYSIS:")
print("=" * 40)

# Get original tokenized lengths for ALL wrong predictions
all_wrong_pred_lengths = []
for pred in all_wrong_preds:
    original_text = str(pred["text"])
    original_tokens = tokenize(original_text)
    original_length = len(original_tokens)
    all_wrong_pred_lengths.append(
        {
            "index": pred["index"],
            "original_length": original_length,
            "truncated": original_length > SEQUENCE_LENGTH,
            "actual_sentiment": pred["actual_sentiment"],
            "predicted_sentiment": pred["predicted_sentiment"],
            "confidence": pred["confidence"],
        }
    )

# Count how many were truncated
total_wrong = len(all_wrong_pred_lengths)
truncated_count = sum(1 for pred in all_wrong_pred_lengths if pred["truncated"])

print(f"Total wrong predictions: {total_wrong}")
print(
    f"Wrong predictions that were truncated: {truncated_count} ({truncated_count / total_wrong * 100:.1f}%)"
)

# Calculate statistics for all test samples (if not done already)
if "all_lengths" not in locals():
    all_lengths = []
    for i in range(len(ds_test)):
        original_text = ds_test[i]["text"]
        original_tokens = tokenize(original_text)
        all_lengths.append(len(original_tokens))

all_truncated = sum(1 for length in all_lengths if length > SEQUENCE_LENGTH)
avg_length_all = sum(all_lengths) / len(all_lengths)
avg_length_wrong = sum(
    pred["original_length"] for pred in all_wrong_pred_lengths
) / len(all_wrong_pred_lengths)

print(f"\nComparison:")
print(f"Overall dataset truncation rate: {all_truncated / len(all_lengths) * 100:.1f}%")
print(f"Wrong predictions truncation rate: {truncated_count / total_wrong * 100:.1f}%")

print(f"\nLength statistics:")
print(f"Overall dataset average length: {avg_length_all:.1f} tokens")
print(f"Wrong predictions average length: {avg_length_wrong:.1f} tokens")

difference = (truncated_count / total_wrong) - (all_truncated / len(all_lengths))
print(f"\nTruncation rate difference: {difference * 100:.1f} percentage points")

if difference > 0.1:
    print("✓ Wrong predictions are MORE LIKELY to be from truncated reviews")
    print("   This suggests truncation is contributing to prediction errors")
elif difference < -0.1:
    print("✓ Wrong predictions are LESS LIKELY to be from truncated reviews")
    print("   This suggests truncation is NOT the main problem")
else:
    print("✓ Wrong predictions have similar truncation rates to overall dataset")
    print("   This suggests truncation is not strongly related to prediction errors")

# Show some examples of the longest wrong predictions
print(f"\nLongest wrong predictions:")
longest_wrong = sorted(
    all_wrong_pred_lengths, key=lambda x: x["original_length"], reverse=True
)[:5]
for i, pred in enumerate(longest_wrong):
    truncated_str = "TRUNCATED" if pred["truncated"] else "NOT TRUNCATED"
    print(
        f"{i + 1}. Index {pred['index']}: {pred['original_length']} tokens ({truncated_str})"
    )
    print(
        f"   Actual: {pred['actual_sentiment']}, Predicted: {pred['predicted_sentiment']}"
    )
    print(f"   Confidence: {pred['confidence']:.4f}")
    print()


CORRECTED LENGTH ANALYSIS:
Total wrong predictions: 122
Wrong predictions that were truncated: 13 (10.7%)

Comparison:
Overall dataset truncation rate: 8.4%
Wrong predictions truncation rate: 10.7%

Length statistics:
Overall dataset average length: 238.9 tokens
Wrong predictions average length: 252.0 tokens

Truncation rate difference: 2.3 percentage points
✓ Wrong predictions have similar truncation rates to overall dataset
   This suggests truncation is not strongly related to prediction errors

Longest wrong predictions:
1. Index 213: 1037 tokens (TRUNCATED)
   Actual: Positive, Predicted: Negative
   Confidence: 0.4118

2. Index 985: 1029 tokens (TRUNCATED)
   Actual: Negative, Predicted: Positive
   Confidence: 0.9030

3. Index 164: 1021 tokens (TRUNCATED)
   Actual: Positive, Predicted: Negative
   Confidence: 0.1794

4. Index 259: 854 tokens (TRUNCATED)
   Actual: Positive, Predicted: Negative
   Confidence: 0.0638

5. Index 201: 798 tokens (TRUNCATED)
   Actual: Positive, Pred